In [ ]:
# PYNQ-Z2: ADS1299 -> UART stream (+ recording of unfiltered samples to recordings/<RECORD_NAME>.npz)
import os, time, subprocess
import numpy as np
from scipy.signal import iirnotch, butter, lfilter, lfilter_zi
from pynq import Overlay, MMIO

# ---------------------------- settings ----------------------------
BITFILE      = 'ads1299.bit'
MODE         = "input"            # "input", "short" or "test"
GAIN         = 24
UART_PORT    = '/dev/ttyPS0'
STREAM_RAW   = True               # True: send unfiltered samples (live_demo.py); False: send filtered samples
BAUD         = 115200
BATCH        = 10                 # samples per UART write
STOP_CONSOLE = True               # free ttyPS0 from the Linux console
CLK_DIV      = 100                # SPI clock divider

RECORD_SEC   = 600                # seconds to record (0 = stream only)
RECORD_NAME  = 'move_01'          # output file name
RECORD_DIR   = 'recordings'
NOTE         = 'V2 on IN1P (+), V1 on SRB1 (-), sitting still'   # stored in the file

VREF = 4.5
FS   = 250                        # 500 SPS is too fast for this Python loop
FRAME_BYTES = 27                  # status + 8 channels x 3 bytes

# ---------------------------- filters (used when STREAM_RAW = False) ----------------------------
b_notch, a_notch = iirnotch(50.0, 30.0, FS)                       # 50 Hz notch, Q30
z_notch = lfilter_zi(b_notch, a_notch) * 0.0
b_bp, a_bp = butter(2, [0.5, 40.0], btype='bandpass', fs=FS)      # 0.5 to 40 Hz band-pass
z_bp = lfilter_zi(b_bp, a_bp) * 0.0

def apply_digital_filters(sample_volts):
    global z_notch, z_bp
    out_notch, z_notch = lfilter(b_notch, a_notch, [sample_volts], zi=z_notch)
    out_bp, z_bp = lfilter(b_bp, a_bp, out_notch, zi=z_bp)
    return out_bp[0]

# ---------------------------- overlay and SPI driver ----------------------------
ol = Overlay(BITFILE)
print("IPs in design:", list(ol.ip_dict.keys()))
spi_name = next((k for k in ol.ip_dict if 'axi_spi_top' in k), None)
if spi_name is None:
    raise RuntimeError("axi_spi_top not found in ip_dict")
BASE = ol.ip_dict[spi_name]['phys_addr']
spi = MMIO(BASE, 0x1000)
rd, wr = spi.read, spi.write
print(f"{spi_name} at {hex(BASE)} | STATUS = {hex(rd(0x04))}")

ADDR_CTRL, ADDR_STATUS, ADDR_TXDATA, ADDR_RXDATA = 0x00, 0x04, 0x08, 0x0C

def ctrl(cs=False, start=False, clk_div=CLK_DIV):
    return (1 | (0 << 1) | (1 << 2) | (int(start) << 3) | (clk_div << 8) | (int(cs) << 16))

W_CS_LOW, W_CS_HIGH, W_START = ctrl(cs=True), ctrl(cs=False), ctrl(cs=True, start=True)

def xfer(b):                      # one SPI byte
    wr(ADDR_TXDATA, b & 0xFF)
    wr(ADDR_CTRL, W_START)
    n = 0
    while rd(ADDR_STATUS) & 1:
        n += 1
        if n > 1_000_000:
            raise TimeoutError("SPI core stuck busy")
    return rd(ADDR_RXDATA) & 0xFF

def cs_low():  wr(ADDR_CTRL, W_CS_LOW)
def cs_high(): wr(ADDR_CTRL, W_CS_HIGH)

def command(op):
    cs_low(); xfer(op); cs_high(); time.sleep(0.001)

def rreg(addr, n=1):
    cs_low(); xfer(0x20 | addr); xfer(n - 1)
    out = [xfer(0x00) for _ in range(n)]
    cs_high()
    return out

def wreg(addr, vals):
    cs_low(); xfer(0x40 | addr); xfer(len(vals) - 1)
    for v in vals:
        xfer(v)
    cs_high(); time.sleep(0.001)

cs_high(); time.sleep(0.05)
command(0x11)                                    # SDATAC
print("ADS1299 ID register:", hex(rreg(0x00)[0]))

# ---------------------------- ADS1299 configuration ----------------------------
GAIN_BITS = {1: 0x00, 2: 0x10, 4: 0x20, 6: 0x30, 8: 0x40, 12: 0x50, 24: 0x60}
MUX_BITS  = {"input": 0x00, "short": 0x01, "test": 0x05}

def to_volts(code):
    return code * VREF / (GAIN * (2**23 - 1))

def configure(mode):
    command(0x0A)                                # STOP
    command(0x11)                                # SDATAC
    time.sleep(0.01)
    wreg(0x01, [0x95 if FS == 500 else 0x96])    # CONFIG1: 0x95 = 500 SPS, 0x96 = 250 SPS
    wreg(0x02, [0xD0 if mode == "test" else 0xC0])
    wreg(0x03, [0xEC])                           # CONFIG3: bias amplifier and reference on
    time.sleep(0.2)
    wreg(0x05, [GAIN_BITS[GAIN] | MUX_BITS[mode]])   # CH1SET
    wreg(0x06, [0x81] * 7)                       # CH2 to CH8 off
    wreg(0x0D, [0x01])                           # BIAS_SENSP: CH1P
    wreg(0x0E, [0x01])                           # BIAS_SENSN: CH1N
    wreg(0x15, [0x20])                           # MISC1: SRB1 to the negative inputs
    print("CONFIG1-3:", [hex(x) for x in rreg(0x01, 3)],
          "| CH1SET:", hex(rreg(0x05)[0]),
          "| BIAS:", hex(rreg(0x0D)[0]), hex(rreg(0x0E)[0]),
          "| MISC1:", hex(rreg(0x15)[0]))
    command(0x10)                                # RDATAC
    command(0x08)                                # START
    print(f"Streaming: mode={mode}, gain={GAIN}, {FS} SPS, bias drive on")

def read_ch1(timeout=1.0):
    t0, n = time.time(), 0
    while rd(ADDR_STATUS) & 4:                   # wait for DRDY
        n += 1
        if n % 2000 == 0 and time.time() - t0 > timeout:
            raise TimeoutError("DRDY timeout")
    cs_low()
    raw = [xfer(0x00) for _ in range(FRAME_BYTES)]
    cs_high()
    ok = (raw[0] & 0xF0) == 0xC0                 # valid status byte
    v = (raw[3] << 16) | (raw[4] << 8) | raw[5]
    if v & 0x800000:
        v -= 1 << 24                             # sign-extend 24 bit
    return to_volts(v), ok, v

configure(MODE)

# ---------------------------- UART ----------------------------
if STOP_CONSOLE:
    subprocess.run(['systemctl', 'stop', 'serial-getty@ttyPS0.service'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['stty', '-F', UART_PORT, str(BAUD), 'raw', '-echo'], check=True)
ser = open(UART_PORT, 'wb', buffering=0)
print(f"\nStreaming {'unfiltered' if STREAM_RAW else 'filtered'} CH1 in microvolts to {UART_PORT} at {BAUD} baud.")

# ---------------------------- recording ----------------------------
n_rec    = int(RECORD_SEC * FS)
rec_code = np.zeros(n_rec, np.int32)            # raw ADC codes
rec_t    = np.zeros(n_rec, np.float64)          # read times, to check the real sampling rate
rec_bad  = []                                    # indices of skipped bad frames
k_rec    = 0
if n_rec:
    print(f"Recording {RECORD_SEC} s of unfiltered samples to {RECORD_DIR}/{RECORD_NAME}.npz")

def save_recording(n):
    os.makedirs(RECORD_DIR, exist_ok=True)
    code, t = rec_code[:n], rec_t[:n]
    uv = code * (VREF / (GAIN * (2**23 - 1))) * 1e6
    path = os.path.join(RECORD_DIR, RECORD_NAME + '.npz')
    np.savez(path, code=code, t=t, bad_at=np.array(rec_bad, np.int64), fs=FS, gain=GAIN, vref=VREF, mode=MODE,
             note=NOTE)
    # quality summary
    dt = np.diff(t); dur = t[-1] - t[0]
    sec = [np.mean(uv[i:i + FS]) for i in range(0, n - FS + 1, FS)]          # 1 s means (mains cancels)
    seg = [uv[i:i + FS] - uv[i:i + FS].mean() for i in range(0, n - FS + 1, FS)]
    ph = np.exp(-2j * np.pi * 50 * np.arange(FS) / FS)
    mains = np.median([2 * np.abs(np.mean(q * ph)) for q in seg]) if seg else 0.0   # 50 Hz amplitude (uV)
    print(f"\nSaved {path}: {n} samples over {dur:.1f} s -> effective rate {(n - 1) / dur:.1f} Hz "
          f"(target {FS}) | bad frames {len(rec_bad)} | gaps > 1.5 sample periods: {(dt > 1.5 / FS).sum()}, max gap {dt.max() * 1e3:.1f} ms")
    print(f"  50 Hz mains amplitude {mains:.0f} uV (ECG is ~1000 uV; aim for < 500)")
    print(f"  DC level (mean) {np.mean(uv):.0f} uV | drift of 1 s means (5-95%) {np.ptp(np.percentile(sec, [5, 95])):.0f} uV | "
          f"peak-to-peak {np.ptp(uv):.0f} uV | clipped samples {(np.abs(code) >= 2**23 - 2).sum()}")

# ---------------------------- main loop ----------------------------
buf = bytearray()
in_batch = bad = total = 0
t_rep, n_rep = time.time(), 0

try:
    while True:
        v, ok, code = read_ch1()
        if not ok:
            bad += 1
            if k_rec < n_rec:
                rec_bad.append(k_rec)
            continue
        if k_rec < n_rec:                        # record the unfiltered sample
            rec_code[k_rec] = code; rec_t[k_rec] = time.time(); k_rec += 1
            if k_rec == n_rec:
                save_recording(k_rec)
                break                            # stops streaming too when the recording is complete

        v_filtered = apply_digital_filters(v)
        buf += b"%d\n" % int(round((v if STREAM_RAW else v_filtered) * 1e6))   # microvolts, one per line
        in_batch += 1
        total += 1
        if in_batch >= BATCH:
            ser.write(buf)
            buf.clear()
            in_batch = 0

        now = time.time()
        if now - t_rep >= 5.0:
            print(f"{(total - n_rep) / (now - t_rep):.0f} SPS | total {total} | bad frames {bad}")
            t_rep, n_rep = now, total

except KeyboardInterrupt:
    print("\nStopped.")
    if 0 < k_rec < n_rec:
        save_recording(k_rec)                    # keep the partial recording
except TimeoutError as e:
    print("\nStopped:", e)
finally:
    try:
        ser.close()
    except Exception:
        pass
    command(0x0A)                                # STOP
    command(0x11)                                # SDATAC
    cs_high()